In [ ]:
import pandas as pd
import arcpy
from utils import *

census_url = r'https://maps.trpa.org/server/rest/services/Demographics/MapServer/28'
taz_url = r'https://maps.trpa.org/server/rest/services/Transportation_Planning/MapServer/6'
census_geom_url = r'https://maps.trpa.org/server/rest/services/Demographics/MapServer/27'

In [ ]:
census_data = get_fs_data(census_url)
taz_data = get_fs_data_spatial(taz_url)
census_geom_data = get_fs_data_spatial(census_geom_url)

In [ ]:
# filter census geom data to only include Geography = 'tract' and census_geom_year = 2020
census_tract = census_geom_data[(census_geom_data['GEOGRAPHY'] == 'Tract') & (census_geom_data['YEAR'] == 2020)]


In [ ]:
census_tract_fc = "memory/census_tract"
taz_fc = "memory/taz"

# Clear any prior runs
for fc in [census_tract_fc, taz_fc]:
    if arcpy.Exists(fc):
        arcpy.management.Delete(fc)

census_tract.spatial.to_featureclass(location=census_tract_fc)
taz_data.spatial.to_featureclass(location=taz_fc)

print(f"census_tract: {arcpy.management.GetCount(census_tract_fc)[0]} features")
print(f"taz:                {arcpy.management.GetCount(taz_fc)[0]} features")

In [ ]:
taz_block_groups = dominant_spatial_join(taz_fc, "TAZ", census_tract_fc, "TRPAID")
taz_block_groups.head()

In [ ]:
# filter census data to only where year_sample = 2024 and variable code = B08119_055E
work_from_home = census_data[(census_data['year_sample'] == 2024) & (census_data['variable_code'] == 'B08119_055E')]

In [ ]:
taz_block_groups['work_from_home'] = taz_block_groups['TRPAID'].map(work_from_home.set_index('TRPAID')['value'])

In [ ]:
tract_taz_pct = pct_overlap(census_tract_fc, "TRPAID", taz_fc, "TAZ")
tract_taz_pct.head()

In [ ]:
taz_work_from_home = (
    tract_taz_pct
    .merge(work_from_home[["TRPAID", "value"]], on="TRPAID", how="left")
    .assign(value_in_taz=lambda d: d["value"] * d["pct_in_taz"] / 100)
    .groupby("TAZ", as_index=False)["value_in_taz"]
    .sum()
    .rename(columns={"value_in_taz": "work_from_home"})
)
taz_work_from_home.head()
taz_work_from_home['work_from_home'] = taz_work_from_home['work_from_home'].round().fillna(0)
taz_work_from_home.head()

In [ ]:
import shutil
from pathlib import Path
local_path = Path().absolute()
PROCESSED = local_path / "data" / "processed_data"

# TAZ lookup: index by int for reliable join
wfh_lookup = taz_work_from_home.set_index(taz_work_from_home["TAZ"].astype(int))["work_from_home"]

alt_folders = [f for f in PROCESSED.iterdir() if f.is_dir() and f.name.startswith("Alternative")]

for src in sorted(alt_folders):
    dst = src.parent / (src.name + "_wfh")

    # Copy the whole folder (overwrite if already exists)
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

    # Adjust SocioEcon_Summer.csv
    socio = dst / "SocioEcon_Summer.csv"
    df = pd.read_csv(socio)
    df["emp_other"] = (
        df["emp_other"]
        + df["taz"].astype(int).map(wfh_lookup).fillna(0)
    ).round(4)
    df.to_csv(socio, index=False)

    print(f"{src.name}  ->  {dst.name}")

print("Done.")

In [ ]:
import pathlib
local_path = pathlib.Path().absolute()
# Save the work from home data to a CSV file
taz_path = r'data/inputs/taz_work_from_home.csv'
taz_path = local_path.parents[0].joinpath(taz_path)
taz_work_from_home.to_csv(taz_path, index=False)

In [ ]:

# read in travel demand model base year parcel data - CHANGE With new name
parcel_base_tdm  = r"Base\data\processed_data\parcel_spatial_joins.parquet"
parcel_base_path = local_path.parents[1].joinpath(parcel_base_tdm)
# read in base year parcel data
sdf_parcel_base  = pd.read_parquet(parcel_base_path)
#output_gdb = r"C:\GIS\Scratch.gdb"

In [ ]:
# Build TRACT ID: state+county+tract (first 11 chars) + year (last 4 chars),
# dropping the block group digit at position 12
sdf_parcel_base["TRACT"] = (
    sdf_parcel_base["BLOCK_GROUP"].str[:11] + sdf_parcel_base["BLOCK_GROUP"].str[-4:]
)

# Sum residential units per (TRACT, TAZ) pair
tract_taz_units = (
    sdf_parcel_base
    .groupby(["TRACT", "TAZ"])["Residential_Units_Adjusted"]
    .sum()
    .reset_index()
)

# Calculate each TAZ's share of its tract's total residential units
tract_totals = tract_taz_units.groupby("TRACT")["Residential_Units_Adjusted"].sum().rename("tract_total")
tract_taz_units = tract_taz_units.join(tract_totals, on="TRACT")
tract_taz_units["pct_in_taz"] = (
    tract_taz_units["Residential_Units_Adjusted"] / tract_taz_units["tract_total"] * 100
).round(4)

# Proportionally assign work-from-home jobs to each TAZ and sum
wfh_lookup = work_from_home.set_index("TRPAID")["value"]

taz_wfh_parcel = (
    tract_taz_units
    .assign(wfh_value=lambda d: d["TRACT"].map(wfh_lookup) * d["pct_in_taz"] / 100)
    .groupby("TAZ", as_index=False)["wfh_value"]
    .sum()
    .rename(columns={"wfh_value": "work_from_home"})
)
# round and make work from home as an int
taz_wfh_parcel["work_from_home"] = taz_wfh_parcel["work_from_home"].round().astype(int)
taz_wfh_parcel.head()

In [ ]:
import shutil
from pathlib import Path
local_path = Path().absolute()

PROCESSED = local_path.parents[0] / "data" / "processed_data"

# TAZ lookup: index by int for reliable join
wfh_lookup = taz_wfh_parcel.set_index(taz_wfh_parcel["TAZ"].astype(int))["work_from_home"]

alt_folders = [f for f in PROCESSED.iterdir() if f.is_dir() and f.name.startswith("Alternative")]

for src in sorted(alt_folders):
    dst = src.parent / (src.name + "_wfh")

    # Copy the whole folder (overwrite if already exists)
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

    # Adjust SocioEcon_Summer.csv
    socio = dst / "SocioEcon_Summer.csv"
    df = pd.read_csv(socio)
    df["emp_other"] = (
        df["emp_other"]
        + df["taz"].astype(int).map(wfh_lookup).fillna(0)
    ).round(4)
    df["emp_other"] = df["emp_other"].astype(int)
    df.to_csv(socio, index=False)

    print(f"{src.name}  ->  {dst.name}")

print("Done.")

In [ ]:
taz_wfh_parcel.to_csv('taz_wfh_parcel.csv', index=False)